In [137]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,Trainer
from sklearn.metrics import f1_score, precision_score, recall_score
import numpy as np


In [152]:
dataset = load_dataset('go_emotions', 'simplified')
num_labels = len(dataset['train'].features['labels'].feature.names)


In [153]:
label_names = dataset["train"].features["labels"].feature.names
print(label_names)

['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


In [139]:
dataset['train'][0]

{'text': "My favourite food is anything I didn't have to cook myself.",
 'labels': [27],
 'id': 'eebbqej'}

In [140]:
def multi_vector(x):
    labels = x['labels']
    multi_vec = np.zeros(num_labels,dtype=np.float32)
    multi_vec[labels] = 1
    x['labels'] = multi_vec
    return x
dataset = dataset.map(multi_vector)

In [141]:
dataset['train'][0]

{'text': "My favourite food is anything I didn't have to cook myself.",
 'labels': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1],
 'id': 'eebbqej'}

In [142]:
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(x):
    return tokenizer(x['text'], truncation=True, padding='max_length', max_length=128)
dataset = dataset.map(tokenize, batched=True)

from datasets import Sequence, Value
dataset = dataset.cast_column("labels", Sequence(Value("float32")))

dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
)


In [143]:
dataset['train'][0]

{'labels': tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]),
 'input_ids': tensor([ 101, 2026, 8837, 2833, 2003, 2505, 1045, 2134, 1005, 1056, 2031, 2000,
         5660, 2870, 1012,  102,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,   

In [144]:
model = AutoModelForSequenceClassification.from_pretrained(model_name,
 num_labels=num_labels, problem_type="multi_label_classification")

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    fp16=True
)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 831.31it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_D

In [145]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits))
    preds = (probs > 0.5).int().numpy()

    labels = labels.astype(int)

    return {
        "micro_f1": f1_score(labels, preds, average="micro"),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
        "precision": precision_score(labels, preds, average="micro"),
        "recall": recall_score(labels, preds, average="micro"),
    }

In [146]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    compute_metrics=compute_metrics)

In [147]:
trainer.train()

Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Weighted F1,Precision,Recall
1,0.094435,0.088236,0.546352,0.328881,0.486873,0.713961,0.442476
2,0.079878,0.083327,0.565151,0.405149,0.531933,0.723581,0.463636
3,0.070936,0.083551,0.576819,0.417969,0.544978,0.708609,0.486364


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.28it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=8142, training_loss=0.09212242933257854, metrics={'train_runtime': 487.6626, 'train_samples_per_second': 267.049, 'train_steps_per_second': 16.696, 'total_flos': 4314807064442880.0, 'train_loss': 0.09212242933257854, 'epoch': 3.0})

In [148]:
metrics = trainer.evaluate(dataset['test'])
print(metrics)

{'eval_loss': 0.08259837329387665, 'eval_micro_f1': 0.5668654993265345, 'eval_macro_f1': 0.40508971785654097, 'eval_weighted_f1': 0.5354388901417848, 'eval_precision': 0.7247232472324723, 'eval_recall': 0.46547637857481433, 'eval_runtime': 4.7669, 'eval_samples_per_second': 1138.47, 'eval_steps_per_second': 71.325, 'epoch': 3.0}


In [149]:
trainer.save_model("./emotion_model")
tokenizer.save_pretrained("./emotion_model")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.53it/s]


('./emotion_model\\tokenizer_config.json', './emotion_model\\tokenizer.json')

In [ ]:
from transformers import pipeline
import torch
import numpy as np

classifier = pipeline(
    "text-classification",
    model="./emotion_model",
    tokenizer="./emotion_model",
    top_k=None
)

text = "really glad you are here"
outputs = classifier(text)[0]  # list of dicts

# Convert LABEL_i → emotion name
label_map = {f"LABEL_{i}": name for i, name in enumerate(label_names)}

# applying threshold 
threshold = 0.25
preds = [(label_map[o['label']], o['score']) for o in outputs if o['score'] > threshold]

print("Predicted Emotions:", preds)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 851.54it/s, Materializing param=pre_classifier.weight]                                  


Predicted Emotions: [('joy', 0.7446218729019165)]


In [167]:
import pickle
with open("emotion_model.pkl", "wb") as f:
    pickle.dump({"model": model, "tokenizer": tokenizer, "label_map": label_map}, f)

print("Model pickled successfully!")

Model pickled successfully!
